# Read all waves -> filter data 

In [ ]:
import pandas as pd
import os

# Define the base path
base_path = "./data-all-waves"

# Create dictionaries to store original and filtered dataframes
dfs = {}
filtered_dfs = {}

# Define the years
years = ['10', '13', '15', '17', '19', '21']

# Read and filter each file
for year in years:
    file_name = f"epcg{year}.sas7bdat"
    file_path = os.path.join(base_path, file_name)
    
    print(f"\nProcessing year 20{year}:")
    
    try:
        # Read with latin1 encoding
        dfs[f'df_{year}'] = pd.read_sas(file_path, encoding='latin1')
        
        # Filter the data
        filtered_dfs[f'df_{year}_filtered'] = dfs[f'df_{year}'][
            (dfs[f'df_{year}']['LFSTAT'] == '1') & 
            (dfs[f'df_{year}']['EMUS'] == 'Y') & 
            ((dfs[f'df_{year}']['DGRDG'] == '1') | (dfs[f'df_{year}']['DGRDG'] == '2'))
        ]
        
        # Print information about the filtering
        print(f"Original rows: {len(dfs[f'df_{year}'])}")
        print(f"Filtered rows: {len(filtered_dfs[f'df_{year}_filtered'])}")
        print(f"Percentage kept: {(len(filtered_dfs[f'df_{year}_filtered'])/len(dfs[f'df_{year}'])*100):.2f}%")
        
    except Exception as e:
        print(f"Error processing data for 20{year}: {str(e)}")

In [ ]:
dfs[f'df_{10}']['D2DG'].unique()

In [ ]:
# Function to verify filtering for any year
def verify_filtering(year):
    df = filtered_dfs[f'df_{year}_filtered']
    print(f"\nVerification for 20{year}:")
    print("LFSTAT values:", df['LFSTAT'].value_counts())
    print("EMUS values:", df['EMUS'].value_counts())
    print("DGRDG values:", df['DGRDG'].value_counts())

# Run verification for each year
for year in years:
    try:
        verify_filtering(year)
    except Exception as e:
        print(f"Error verifying 20{year}: {str(e)}")

# Select variables (and make the names consistent)

In [ ]:
# Define variables that are consistent across all waves
common_vars = ['REFYR', 'COHORT', 'AGE', 'CTZN', 'GENDER', 'RACEM', 'HRSWK', 'SALARY', 
               'MGRNAT', 'MGROTH', 'MGRSOC', 'BAYR', 'DGRYR', 'D2YR', 'D3YR', 'D4YR', 'D5YR',
               'BAMN', 'HDMN', 'D2MN', 'D3MN', 'D4MN', 'D5MN', 'DGRDG', 'D2DG', 'D3DG', 'D4DG', 'D5DG',
               'NBAMEMG', 'NBAMENG', 'NBASEMG', 'NBANENG', 'NDGMEMG', 'NDGMENG',
               'ND2MEMG', 'ND2MENG', 'ND2SEMG', 'ND2NENG', 'ND3MEMG', 'ND3MENG',
               'EMSIZE', 'EMSECSM', 'EMSECDT', 'BSDGN', 'BADGRUS', 'LFSTAT', 'EMUS',
               'STRTMN', 'STRTYR', 'LWMN', 'LWYR', 'TCDGCMP', 'HSYR', 'NRREA']

# Define variable mappings for those that change names
var_mappings = {
    'pre_2019': {
        'NBAMED': 'NBAMED',
        'NBANED': 'NBANED',
        'NDGRMED': 'NDGRMED',
        'ND2MED': 'ND2MED',
        'ND2NED': 'ND2NED',
        'ND3MED': 'ND3MED'
    },
    'post_2019': {
        'NBAMED': 'N2BAMED',
        'NBANED': 'N2BANED',
        'NDGRMED': 'N2DGRMED',
        'ND2MED': 'N2D2MED',
        'ND2NED': 'N2D2NED',
        'ND3MED': 'N2D3MED'
    }
}

# Create dictionary to store selected dataframes
selected_dfs = {}

# Process each year
for year in years:
    print(f"\nProcessing year 20{year}:")
    
    try:
        # Determine which variable names to use based on year
        if int(year) < 19:  # Before 2019
            year_specific_vars = var_mappings['pre_2019']
        else:  # 2019 and after
            year_specific_vars = var_mappings['post_2019']
        
        # Combine common variables with year-specific variables
        all_vars = common_vars + list(year_specific_vars.values())
        
        # Select only variables that exist in the dataset
        existing_vars = [var for var in all_vars if var in filtered_dfs[f'df_{year}_filtered'].columns]
        
        # Select variables
        selected_dfs[f'df_{year}_selected'] = filtered_dfs[f'df_{year}_filtered'][existing_vars]
        
        # Print information
        print(f"Original columns: {len(filtered_dfs[f'df_{year}_filtered'].columns)}")
        print(f"Selected columns: {len(existing_vars)}")
        print(f"Rows: {len(filtered_dfs[f'df_{year}_filtered'])}")
        
        # Print any variables that were in our list but not found in the dataset
        missing_vars = set(all_vars) - set(existing_vars)
        if missing_vars:
            print(f"Warning: The following variables were not found in 20{year} data:")
            print(missing_vars)
            
    except Exception as e:
        print(f"Error processing data for 20{year}: {str(e)}")

# Verify the selected variables for one year (e.g., 2010)
print("\nSelected variables for 2010:")
print(selected_dfs['df_10_selected'].columns.tolist())

# Merge all waves into one dataframe

In [ ]:
 # First, let's standardize the variable names (converting post-2019 names to pre-2019 format)
def standardize_names(df, year):
    if int(year) >= 19:  # For 2019 and 2021
        rename_dict = {
            'N2BAMED': 'NBAMED',
            'N2BANED': 'NBANED',
            'N2DGRMED': 'NDGRMED',
            'N2D2MED': 'ND2MED',
            'N2D2NED': 'ND2NED',
            'N2D3MED': 'ND3MED'
        }
        return df.rename(columns=rename_dict)
    return df

# Create a list to store standardized dataframes
standardized_dfs = []

# Process each year
for year in years:
    # Get the selected dataframe for this year
    df = selected_dfs[f'df_{year}_selected'].copy()
    
    # Add a year identifier column
    df['SURVEY_YEAR'] = f'20{year}'
    
    # Standardize variable names
    df = standardize_names(df, year)
    
    # Append to list
    standardized_dfs.append(df)
    
    print(f"Processed 20{year}: {df.shape[0]} rows, {df.shape[1]} columns")

# Merge all dataframes
merged_df = pd.concat(standardized_dfs, axis=0, ignore_index=True)

# Print information about the merged dataset
print("\nMerged dataset information:")
print(f"Total rows: {merged_df.shape[0]}")
print(f"Total columns: {merged_df.shape[1]}")
print("\nNumber of observations by survey year:")
print(merged_df['SURVEY_YEAR'].value_counts().sort_index())

# Verify that all variable names are consistent
print("\nAll columns in merged dataset:")
print(sorted(merged_df.columns.tolist()))

# Optional: Check for any missing values in key variables
print("\nMissing values in key variables:")
print(merged_df[['SURVEY_YEAR', 'REFYR', 'AGE', 'SALARY']].isnull().sum())

# Convert REFYR from float strings ('2010.0') to integers (2010)
merged_df['REFYR'] = merged_df['REFYR'].astype(float).astype(int)

# Verify the conversion
print("\nREFYR values after conversion:")
print(merged_df['REFYR'].unique())


merged_df = merged_df.drop('SURVEY_YEAR', axis=1)
print("\nSURVEY_YEAR removed as it matches REFYR")

In [ ]:
merged_df

# Export to .csv

In [ ]:
# List of variables that should be integers
int_vars = ['AGE', 'HRSWK', 'SALARY', 'BAYR', 'DGRYR', 'D2YR', 'D3YR', 'D4YR', 'D5YR', 
            'BAMN', 'HDMN', 'D2MN', 'D3MN', 'D4MN', 'D5MN', 'BSDGN', 
            'STRTMN', 'STRTYR', 'LWMN', 'LWYR', 'HSYR']

# Check current dtypes
print("Current data types:")
for var in int_vars:
    if var in merged_df.columns:
        print(f"{var}: {merged_df[var].dtype}")

# Convert to integers
for var in int_vars:
    if var in merged_df.columns:
        try:
            # First convert to float (in case there are decimal points)
            # Then convert to integer
            merged_df[var] = merged_df[var].astype(float).astype(int)
        except Exception as e:
            print(f"Error converting {var}: {str(e)}")
            # Print some sample values that might be causing issues
            print(f"Sample problematic values in {var}:")
            print(merged_df[var].head())

# Verify conversions
print("\nNew data types after conversion:")
for var in int_vars:
    if var in merged_df.columns:
        print(f"{var}: {merged_df[var].dtype}")

In [ ]:
# Now save to CSV
output_file = os.path.join(base_path, "merged_nscg_2010_to_2021.csv")
merged_df.to_csv(output_file, index=False)

print(f"\nDataset saved to: {output_file}")
print(f"Number of rows: {merged_df.shape[0]}")
print(f"Number of columns: {merged_df.shape[1]}")